In [1]:
!pip install pandas
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Looking in links: /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2020/avx512, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo/avx512, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2020/avx2, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo/avx2, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2020/generic, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo/generic, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/generic
Looking in indexes: https://download.pytorch.org/whl/cu124
Looking in links: /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2020/avx512, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo/avx512, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2020/avx2, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo/avx2, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2020/generic, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo/gene

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import socket
print(socket.gethostname())
import torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader, Dataset
print("cuda available?", torch.cuda.is_available())
import scipy.linalg as linalg
import matplotlib.pyplot as plt
print(f"NumPy:  {np.__version__}")
print(f"PyTorch: {torch.__version__}")
import os
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

fc10405
cuda available? True
NumPy:  1.25.2
PyTorch: 2.6.0+cu124
Using cuda device


In [3]:
trans_scin_range = (0.01,0.2)
AGN_scin_range = (0.00005,0.2)  ## replicates the observed variability fractions in CHILES VERDES
varbins = np.array([0,0.02,0.1]) ## fractional brightness fluctuation standard deviation. 0.1 Corresponds to a 10% RMS flux density fluctuation
varprobs = np.array([136/185,33/185,16/185]) ## correspond probabilities. Must sum to 1

### LOAD IN THE VARIABILITY: ###

#scinlib = np.load('/Users/dtian/Documents/Programming/Di_Research/Di_codefromFir/scintillation_library.npz')['varlib']
scinlib = np.load('/home/dtian/scratch/Di_Research/Di_codefromFir/scintillation_library.npz')['varlib']
#varlib = np.load('/Users/dtian/Documents/Programming/Di_Research/Di_codefromFir/variation_library.npz')['varlib']
varlib = np.load('/home/dtian/scratch/Di_Research/Di_codefromFir/variation_library.npz')['varlib']

In [4]:
#transientsdata = (np.load('/Users/dtian/Documents/Programming/Di_Research/Di_codefromFir/TDE_afterglows/ISM_profile/JettedTDE_ISM1.npz')['data'])[:,:,20]
transientsdata = (np.load('/home/dtian/scratch/Di_Research/Di_codefromFir/TDE_afterglows/ISM_profile/JettedTDE_ISM1.npz')['data'])[:,:,20]

In [5]:
def gaussian2d(x,y,meanx,meany):

    FWHM = 5

    '''
    returns a 2d gaussian with the right shape
    '''

    return np.exp(-0.5*2.3**2*((x-meanx)**2+(y-meany)**2)/FWHM**2) # factor of 2.3 converts the FWHM to a standard deviation

def genbackground(xlen,ylen,tlen,Nbgnd,noiseamp=0.1):

    x = np.arange(xlen) ## x,y,t arrays
    y = np.arange(ylen)
    t = np.arange(tlen) 

    xx,yy,tt = np.meshgrid(x,y,t) ## create a grid of x,y positions to evaluate at

    rng = np.random.default_rng(123)

    noisefield = noiseamp*rng.standard_normal(size = (xlen,ylen,tlen)) # generate the noise

    backgroundfield = np.zeros(noisefield.shape) ## array for the transient layer to be created in
    
    x0 = [] ## these record the x, y, and time mean values for each transient signal
    y0 = [] ## these record the x, y, and time mean values for each transient signal
    positions = np.zeros((Nbgnd,2))

    for i in range(Nbgnd): ## for each background source

        x0.append(np.random.randint(xlen))
        y0.append(np.random.randint(ylen)) ## generate a uniform random map position

        bgfluxdensity = 10**(2*np.random.rand()-1) ## brightness, right now log-uniform between 0.1 and 10

        layer = bgfluxdensity*gaussian2d(xx,yy,x0[-1],y0[-1]) # add the map layer value

        ## apply intrinsic variability:

        idx = np.random.randint(2000)
        phase = np.random.randint(608)

        randfloat = np.random.rand()
        
        if randfloat <= varprobs[0]:
            amp = varbins[0]
        elif randfloat <= varprobs[0]+varprobs[1]:
            amp = varbins[1]
        else:
            amp = varbins[2]
        
        layer*= 1+amp*(varlib[idx][phase:])[np.newaxis,np.newaxis,:tlen]

        ## apply extrinsic variability:

        idx = np.random.randint(2000)
        phase = np.random.randint(608)
        amp = 10**((np.log10(AGN_scin_range[1]/AGN_scin_range[0]))*np.random.rand()+np.log10(AGN_scin_range[0]))
        #print(amp)

        layer*= 1+amp*(varlib[idx][phase:])[np.newaxis,np.newaxis,:tlen]

        backgroundfield += layer

    positions[:,0] = np.array(x0)
    positions[:,1] = np.array(y0)

    return backgroundfield,noisefield,positions
        
def gentransients(xlen,ylen,tlen,Ntrans):

    global transientsdata

    x = np.arange(xlen) ## x and y arrays
    y = np.arange(ylen)
    t = np.arange(tlen) 

    xx,yy,tt = np.meshgrid(x,y,t) ## create a grid of x,y positions to evaluate at

    transient_field = np.zeros((xlen,ylen,tlen)) ## array for the transient layer to be created in
    
    x0 = [] ## these record the x, y, and time mean values for each transient signal
    y0 = [] ## these record the x, y, and time mean values for each transient signal
    positions = np.zeros((Ntrans,2))

    for i in range(Ntrans):

        x0.append(np.random.randint(xlen))
        y0.append(np.random.randint(ylen))

        LCidx = np.random.randint(7000)

        LC = transientsdata[LCidx]
        LC /= np.max(LC)

        transfluxdensity = 10**(np.random.rand()-0.5) ## right now log-uniform between 0.3 and 3 ## change as you see fit

        ## apply the shape of the lightcurve
        
        layer = transfluxdensity*gaussian2d(xx,yy,x0[-1],y0[-1])*LC[np.newaxis,np.newaxis,:tlen]

        ## apply extrinsic variability:

        idx = np.random.randint(2000)
        phase = np.random.randint(608)
        amp = 10**((np.log10(trans_scin_range[1]/trans_scin_range[0]))*np.random.rand()+np.log10(trans_scin_range[0]))
        #print(amp)

        layer*= 1+amp*(varlib[idx][phase:])[np.newaxis,np.newaxis,:tlen]

        transient_field += layer

    positions[:,0] = np.array(x0)
    positions[:,1] = np.array(y0)

    return transient_field,positions

In [6]:
bg,noise,bgpositions = genbackground(100,100,608,15,noiseamp = 0.1)
#bg: intrinsic and extrinsic, noise: random, bgpositions: 

In [7]:
transients,transientspositions = gentransients(100,100,608,2)

In [8]:
# generates and saves animation frames for the above generated field
for frame in range(100):
    plt.figure(figsize = (5,5))
    plt.imshow((bg+noise+transients)[:,:,6*frame],vmax = 5,vmin = 0)
    plt.xticks([])
    plt.yticks([])
    #plt.savefig('/Users/dtian/Documents/Programming/Di_Research/example_animation/frame_%d.png' %frame)
    #plt.show()
    plt.close()

In [9]:
def genfields(Nfields=10, xlen=100, ylen=100, tlen=608, Nbgnd=15 ,noiseamp=0.1, Ntrans=2):
    combined_fields = []
    backgrounds = []
    noises = []
    transients = []
    bgpositions = []
    transientspositions = []
    for field in range(Nfields):
        bg,noise,bgposition = genbackground(xlen,ylen,tlen,Nbgnd,noiseamp)
        transient,transientsposition = gentransients(xlen,ylen,tlen,Ntrans)
        combined_fields.append(bg+noise+transient)
        backgrounds.append(bg)
        noises.append(noise)
        transients.append(transient)
        bgpositions.append(bgposition)
        transientspositions.append(transientsposition)
    combined_fields = np.array(combined_fields)
    backgrounds = np.array(backgrounds)
    noises = np.array(noises)
    transients = np.array(transients)
    bgpositions = np.array(bgpositions)
    transientspositions = np.array(transientspositions)
    return combined_fields, backgrounds, noises, transients, bgpositions, transientspositions

In [10]:
def normalize_data(data):
    mean = data.mean()
    std = data.std()
    return (data - mean) / std, mean, std

def convert_to_torch_data(data):
    data = torch.tensor(data, dtype=torch.float32)
    data = data.squeeze(0)
    if data.dim() == 4:                       # NHWC -> NCHW
        data = data.permute(0, 3, 1, 2).contiguous()
    data = data.float().to(device)
    return normalize_data(data)

In [11]:
a, b, c, d, e, f= genfields()
X_train,_,_ = convert_to_torch_data(a)
#b,_,_ = convert_to_torch_data(b)
#c,_,_ = convert_to_torch_data(c)
#d,_,_ = convert_to_torch_data(d)
#e,_,_ = convert_to_torch_data(e)
y_train,_,_ = convert_to_torch_data(f)
a, b, c, d, e, f= genfields()
X_test,_,_ = convert_to_torch_data(a)
y_test,_,_ = convert_to_torch_data(f)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

torch.Size([10, 608, 100, 100])
torch.Size([10, 2, 2])
torch.Size([10, 608, 100, 100])
torch.Size([10, 2, 2])


In [12]:
train_dataset = TensorDataset(X_train, y_train)
test_dataset  = TensorDataset(X_test,  y_test)

batch_size = 64

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

In [13]:
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.block(x))  # skip connection


class NeuralNetwork(nn.Module):
    def __init__(self, n_trans=2, n_params=2, tlen=608):
        super().__init__()
        self.n_trans = n_trans
        self.n_params = n_params

        # large kernel stem to capture broad Gaussian spatial extent
        self.stem = nn.Sequential(
            nn.Conv2d(tlen, 96, kernel_size=7, padding=3),
            nn.BatchNorm2d(96),
            nn.ReLU(),
        )

        # (N, 96, 50, 50)
        self.layer1 = nn.Sequential(
            ResBlock(96),
            ResBlock(96),
            ResBlock(96),
            nn.MaxPool2d(2),         # -> (N, 96, 25, 25)
        )

        self.layer2 = nn.Sequential(
            nn.Conv2d(96, 192, kernel_size=3, padding=1),
            nn.BatchNorm2d(192),
            nn.ReLU(),
            ResBlock(192),
            ResBlock(192),
            ResBlock(192),
            nn.MaxPool2d(2),         # -> (N, 192, 12, 12)
        )

        self.layer3 = nn.Sequential(
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.BatchNorm2d(384),
            nn.ReLU(),
            ResBlock(384),
            ResBlock(384),
            ResBlock(384),
            nn.MaxPool2d(2),         # -> (N, 256, 6, 6)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),            # -> (N, 384*6*6 = 13824)
            nn.Linear(13824, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, n_trans * n_params),
        )

    def forward(self, x):
        # x = x.permute(0, 3, 1, 2)   # (N, x, y, t) -> (N, t, x, y)
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.fc(x).view(-1, self.n_trans, self.n_params)


#model = NeuralNetwork(tlen=100).to(device)
#print(model)

In [14]:
'''
The hungarian_loss take account the fact that two similar transient 3D image, (A, B, C) and (B, A, C), would be similar but with differnent loss
Hungarian_loss take care of it and make them similar loss
'''

from scipy.optimize import linear_sum_assignment
import torch.nn.functional as F

def hungarian_loss(pred, target):
    # pred, target: (N, n_trans, n_params)
    batch_size, n_trans = pred.shape[:2]
    total_loss = torch.tensor(0.0, requires_grad=True)

    for i in range(batch_size):
        # pairwise MSE cost matrix between predicted and true transients
        cost = torch.stack([
            torch.stack([F.mse_loss(pred[i, j], target[i, k])
                         for k in range(n_trans)])
            for j in range(n_trans)
        ])
        # find optimal assignment (no gradient needed here)
        row_ind, col_ind = linear_sum_assignment(cost.detach().cpu().numpy())
        # sum loss over matched pairs
        matched = sum(cost[r, c] for r, c in zip(row_ind, col_ind))
        total_loss = total_loss + matched / n_trans

    return total_loss / batch_size

In [15]:
model = NeuralNetwork().to(device)
print(model)
loss_fn = hungarian_loss

NeuralNetwork(
  (stem): Sequential(
    (0): Conv2d(608, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3))
    (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (layer1): Sequential(
    (0): ResBlock(
      (block): Sequential(
        (0): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (relu): ReLU()
    )
    (1): ResBlock(
      (block): Sequential(
        (0): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=

In [16]:
def train_loop(dataloader, model, loss_fn, optimizer):
    model.train()
    total_loss = 0
    for X, y in dataloader:
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    return total_loss / len(dataloader)


def test_loop(dataloader, model, loss_fn):
    model.eval()
    num_batches = len(dataloader)
    test_loss = 0
    all_preds = []

    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            all_preds.append(pred)

    test_loss /= num_batches
    all_preds = torch.cat(all_preds, dim=0)
    return all_preds, test_loss  # return both

In [28]:
X_small = X_train[:2].to(device)
y_small = y_train[:2].to(device)
print(X_small.shape)
print(y_small.shape)

model.train()
pred = model(X_small)
print("Forward:", pred.shape)
loss = hungarian_loss(pred, y_small)
loss.backward()
print("Backward OK, loss:", loss.item())

torch.Size([2, 608, 100, 100])
torch.Size([2, 2, 2])


RuntimeError: mat1 and mat2 shapes cannot be multiplied (2x55296 and 13824x512)

In [ ]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output

learning_rate = 1e-3
batch_size = 64
epochs = 100
train_losses = []
test_losses = []

loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)#, verbose=True)

for t in range(epochs):
    train_loss = train_loop(train_dataloader, model, loss_fn, optimizer)
    preds, test_loss = test_loop(test_dataloader, model, loss_fn)
    scheduler.step(test_loss)
    
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    
    clear_output(wait=True)
    plt.clf() 
    plt.plot(train_losses, label='Train')
    plt.plot(test_losses, label='Test')
    plt.yscale('log')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'Epoch {t+1}, Train: {train_loss:.4f}, Test: {test_loss:.4f}')
    plt.legend()
    plt.show()

print("Done!")